In [1]:
import pymupdf
import os
import json
import re
from lib.helpers.slugify import slugify

# Meta

In [22]:
path = "data/pdf/2022/BHA_20220106.pdf"
doc = pymupdf.open(path)
def get_date(doc):
    page = doc.load_page(0)
    text = page.get_text()
    text = text.split('Manaus, ')[1].replace("\n", "")
    return text

In [ ]:
path = 'data/pdf/2022'
pdfs = sorted(os.listdir(path))
n = 1
for pdf in pdfs:
    mmdd = pdf[8:12]
    pdf_path = f'data/pdf/2022/{pdf}'
    pathJson = f"data/2022/{mmdd}.json"
    doc = pymupdf.open(pdf_path)
    date = get_date(doc)
    boletim = {
         "meta": {
            "pt": {
                "issn": "2965-0291",
                "volume": "02",
                "date": date
            },
            "number": str(n)
        }
    }
    with open(pathJson, "w") as file:
        json.dump(boletim, file, ensure_ascii=False, indent=3)
    n += 1
    print(mmdd, n)

# current_conditions

In [61]:
def current_conditions(path):
    doc = pymupdf.open(path)
    page = doc.load_page(3)
    text = page.get_text()
    text = text.split("Condições atuais\n")[1]
    text = text.split("\n1\nAbacaxis")[0]
    text = text.replace("\n", "")
    return text

In [53]:
path = 'data/pdf/2022/BHA_20220303.pdf'
text = current_conditions(path)
text

'Mapas das condições observadas de precipitação e gráficos individuais por bacias são produzidos a partir dosdados MERGE/GPM gerados pelo INPE/CPTEC, considerando como climatologia o período de 2000 a 2021.Entre  os dias 2 de fevereiro e 3 de março de 2022, ao longo da análise do comportamento das chuvassobre a Bacia Amazônica observado predomínio de deficit (laranja) de precipitação caracterizando abacia hidrográfica dos rios Amazonas em território brasileiro, Aripuanã, Beni, Curuá Una, Guaporé, Iriri,Juruena, Mamoré, Teles Pires e Xingu, excessos de precipitação (azul) registrados sobre as baciashidrográficas dos rios Abacaxis, Amazonas em território peruano, Branco, Coari, Japurá, Javari, Ji-Paraná, Jutaí, Madeira, Marañon, margem esquerda do Amazonas no nordeste do Amazonas e noroestedo Pará, Negro e curso principal do Solimões, bacias do Juruá, margem esquerda do Amazonas nonordeste do Pará, Napo, Purus, Tapajós e Ucayali com volumes de chuva considerados próximos daclimatologia d

In [48]:
def tmp_current_conditions(path):

    path = "data/pdf/2022/BHA_20220106.pdf"
    doc = pymupdf.open(path)
    page = doc.load_page(2)
    text = page.get_text()
    text = text.split("Página 2 de 20\n")[1]
    text = text.split("\n1\nAbacaxis")[0]
    return text

In [62]:
path = 'data/pdf/2022'
pdfs = sorted(os.listdir(path))
n = 1
for pdf in pdfs[49:]:
    mmdd = pdf[8:12]
    pdf_path = f'data/pdf/2022/{pdf}'
    pathJson = f"data/2022/{mmdd}.json"
    # text = tmp_current_conditions(path)
    text = current_conditions(pdf_path)

    with open(pathJson, "r") as file:
        boletin = json.load(file)
    pt = {"pt": text}
    boletin['current_conditions'] = pt
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(mmdd)

1215
1222
1229


# analysis

In [267]:
def clean_text(text: str) -> str:
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [130]:
def get_bacias(path):
    doc = pymupdf.open(path)
    bacias = {}
    for pg in range(3,15):
        page = doc.load_page(pg)
        blocks = page.get_text_blocks()
        for block in blocks:
            x0, y0, x1, y1, text, block_no, block_type = block
            if x0 < 200 and not "Análise" in text:
                if "Valores de Referência" in text or "Previsão multi-modelo" in text:
                    continue
                name = clean_text(text)
                slug_name = slugify(name)                
            if x0 > 250 and not "Bacia Amazônica –" in text:
                bacias[slug_name] = {}
                bacias[slug_name]['name'] = name
                bacias[slug_name]['text'] = clean_text(text)
    return bacias

In [131]:
path = 'data/pdf/2022/BHA_20220106.pdf'   
        
bacias = get_bacias(path)

In [133]:
len(bacias)

32

In [107]:
pathJson = 'data/2022/1215.json'
with open(pathJson, "r") as file:
    boletin = json.load(file)
boletin['bacias'] = bacias
with open(pathJson, "w") as file:
    json.dump(boletin, file, ensure_ascii=False, indent=3)

In [137]:
path = 'data/pdf/2022'
pdfs = sorted(os.listdir(path))
pdfs[:13]

['BHA_20220106.pdf',
 'BHA_20220113.pdf',
 'BHA_20220120.pdf',
 'BHA_20220127.pdf',
 'BHA_20220203.pdf',
 'BHA_20220210.pdf',
 'BHA_20220217.pdf',
 'BHA_20220224.pdf',
 'BHA_20220303.pdf',
 'BHA_20220310.pdf',
 'BHA_20220317.pdf',
 'BHA_20220324.pdf',
 'BHA_20220331.pdf']

In [138]:
for pdf in pdfs[:13]:
    mmdd = pdf[8:12]
    pathPdf = f'data/pdf/2022/{pdf}'
    pathJson = f"data/2022/{mmdd}.json"

    bacias = get_bacias(pathPdf)
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    boletin['bacias'] = bacias
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(mmdd)

0106
0113
0120
0127
0203
0210
0217
0224
0303
0310
0317
0324
0331


In [111]:
jsonFiles[49:]

['1215.json', '1222.json', '1229.json']

In [ ]:
path = 'data/2022'
jsonFiles = sorted(os.listdir(path))
for i in jsonFiles:
    pathJson = f"{path}/{i}"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    bacias = boletin['bacias']
    analysis = []
    for k, v in bacias.items():
        v['id'] = k
        analysis.append(v)
    boletin['analysis'] = analysis
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    
        
    print(i)

In [172]:
def trans_analysis(analysis):
   for i in analysis:
      text = i['text']
      padrao_mim_max = r"variando entre\s+(\d+(?:[.,]\d+)?)\s+e\s+(\d+(?:[.,]\d+)?)\s+mm"
      match = re.search(padrao_mim_max, text, re.IGNORECASE)
      min = match.group(1)
      max = match.group(2)
      i['min'] = min
      i['max'] = max
      padrao_obs = r"foram observados\s+(\d+(?:[.,]\d+)?\s+mm)"
      match_obs = re.search(padrao_obs, text, re.IGNORECASE)
      observados = match_obs.group(1)
      i['observados'] = observados
      padra_ano = r"o valor de\s+(-?\s?\d+(?:[.,]\d+)?)"
      match_ano = re.search(padra_ano, text, re.IGNORECASE)
      anomalia = match_ano.group(1)
      i['anomalia'] = anomalia
      p_class = r"condição de\s+(.*?)\.\s+Nas próximas semanas"
      match_class = re.search(p_class, text, re.IGNORECASE)
      classification = match_class.group(1)

      p_trend = r"comportamento climático indica\s+(.*?)\s+dos volumes"
      match_trend = re.search(p_trend, text, re.IGNORECASE)
      trend = match_trend.group(1)

      p_prognostico = r"sugere um comportamento\s+(.*?)(?:\.|$)"
      match_prognostico = re.search(p_prognostico, text, re.IGNORECASE)
      prognostico = match_prognostico.group(1)
      if re.search(r"^de\s", prognostico):
         prognostico = prognostico[3:]    

      i18n = {
               "pt": {
                  "classification": classification,
                  "trend": trend,
                  "prognostico": prognostico
               }
            }
      i['i18n'] = i18n
   return analysis

In [174]:
path = f'data/2022'
jsonFiles = sorted(os.listdir(path))

In [ ]:
for i in jsonFiles[29:]:
    pathJson = f"{path}/{i}"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    analysis = boletin['analysis']
    analysis = trans_analysis(analysis)
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(pathJson)
    

In [ ]:
for i in jsonFiles:
    pathJson = f"{path}/{i}"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    del boletin['bacias']
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
        print(pathJson)


In [248]:
for i in jsonFiles:
    pathJson = f"{path}/{i}"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    analysis = boletin['analysis']
    for i in analysis:
        del i['text']
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    
    

multimodel

In [269]:
pathPdf = 'data/pdf/2022/BHA_20220310.pdf'
def get_multimodal(pathPdf):
    doc = pymupdf.open(pathPdf)
    page = doc.load_page(14)
    blocks = page.get_text_blocks()
    seven_days = clean_text(blocks[1][4])
    fourteen_days = clean_text(blocks[2][4])
    return seven_days, fourteen_days

In [290]:
jsonFiles[34:]

['0901.json',
 '0908.json',
 '0915.json',
 '0922.json',
 '0929.json',
 '1006.json',
 '1013.json',
 '1020.json',
 '1027.json',
 '1103.json',
 '1110.json',
 '1117.json',
 '1124.json',
 '1201.json',
 '1208.json',
 '1215.json',
 '1222.json',
 '1229.json']

In [291]:
for i in jsonFiles[34:]:
    mmdd = i.replace(".json", "")
    pathJson = f"{path}/{i}"
    pathPdf = f"data/pdf/2022/BHA_2022{mmdd}.pdf"
    seven_days, fourteen_days = get_multimodal(pathPdf)
    multimodel = {
        "pt": {
            "seven_days": seven_days,
            "fourteen_days": fourteen_days
        }
    }
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    boletin['multimodel'] = multimodel

    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(mmdd)


0901
0908
0915
0922
0929
1006
1013
1020
1027
1103
1110
1117
1124
1201
1208
1215
1222
1229


In [286]:
pathPdf = f"data/pdf/2022/BHA_20220901.pdf"
seven_days, fourteen_days = get_multimodal(pathPdf)

In [288]:
fourteen_days

'A Figura a direita, apresenta o prognóstico para o intervalo de 14 dias entre 31/08/2022 e 13/09/2022, com previsão de chuvas abaixo (laranja) dos valores climatológicos sobre áreas do curso principal do Amazonas em território peruano, bacias dos rios Aripuanã, Beni, Branco, Curuá Una, Guaporé, Içá, Iriri, Javari, Juruá, Juruena, Jutaí, Mamoré, Marañon, Negro, Purus, Tapajós, Teles Pires, Ucayali, Xingu e bacias da margem esquerda do Rio Amazonas no nordeste do Amazonas, poderão ser observadas áreas com excesso (azul) de precipitação em relação a climatologia do período sobre as bacias da margem esquerda do Amazonas no nordeste do Amazonas e noroeste do Pará, demais bacias com previsão de chuvas próximas da normalidade do período.'

In [285]:
text.split("A Figura a direita, ")

['O prognóstico de anomalias de precipitação previsto para o intervalo de 07 dias entre 24/08/2022 e 30/08/2022\n(figura a esquerda) indica predomínio de áreas com chuvas próximas a climatologia do período sobre as bacias\nmonitoradas, previsão de deficit (laranja) de precipitação predominando sobre áreas das bacias de captação dos\nrios Abacaxis, Iriri, Juruá, Madeira, Purus, Teles Pires, Ucayali e Xingu, poderão ser observadas áreas com\nexcesso (azul)  de precipitação em relação a climatologia do período sobre o curso principal do Amazonas em\nterritório peruano e bacias dos rios Beni, Branco, Guaporé, Içá, Japurá, Mamoré, Marañon, Napo e bacias da\nmargem esquerda do Amazonas no nordeste do Amazonas e nordeste e noroeste do Pará, demais bacias com\nprevisão de chuvas próximas da normalidade do período. \n',
 'apresenta o prognóstico para o intervalo de 14 dias entre  24/08/2022 e  06/09/2022, com\nprevisão de chuvas abaixo (laranja) dos valores climatológicos sobre o curso principa

# Imagem

In [2]:
import pymupdf 
from lib.images.anomaly import img_anomaly
from lib.images.categorization import img_categorization
from lib.images.reference import img_reference
import os

In [33]:
def img_current_conditions(doc, output_path):

    path = f"{output_path}/current_conditions/"
    page = doc.load_page(3)
    # maps
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['width'] == 800:
            img = i['image'] 
            break
    map = f'{path}map_current_conditions.png'
    with open(map, "wb") as f:
                    f.write(img) # type: ignore
    
    # Tabels
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "1\nAbacaxis" in text:
            break

    top = y0 - 5 # type: ignore
    left = x0 - 15 # type: ignore
    right = x1 + 45 # type: ignore
    bottom = y1 + 5 # type: ignore
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}table_current_conditions.png')

In [ ]:
path_dir = 'data/pdf/2022'
pdfs = sorted(os.listdir(path_dir))
pdfs[:49]

In [30]:
for pdf in pdfs:
    mmdd = pdf[8:12]
    output_path = f"data/images/2022/{mmdd}"
    os.mkdir(output_path)
    os.mkdir(f'{output_path}/analysis')
    os.mkdir(f'{output_path}/anomaly')
    os.mkdir(f'{output_path}/categorization')
    os.mkdir(f'{output_path}/current_conditions')
    os.mkdir(f'{output_path}/multimodel')
    os.mkdir(f'{output_path}/reference')
    # print(mmdd)

FileExistsError: [Errno 17] File exists: 'data/images/2022/0106'

In [34]:
pdfs[49:]

['BHA_20221215.pdf', 'BHA_20221222.pdf', 'BHA_20221229.pdf']

In [35]:
path_dir = 'data/pdf/2022'
pdfs = sorted(os.listdir(path_dir))
for pdf in pdfs[49:]:
    mmdd = pdf[8:12]
    full_path = f'{path_dir}/{pdf}'
    doc = pymupdf.open(full_path)
    output_path = f"data/images/2022/{mmdd}"
    img_current_conditions(doc, output_path)
    
    print(full_path)

data/pdf/2022/BHA_20221215.pdf
data/pdf/2022/BHA_20221222.pdf
data/pdf/2022/BHA_20221229.pdf


In [51]:
from lib.helpers.slugify import slugify
bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
    'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
    'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
    'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
    'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
    'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
    'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']

slug_names = []
for bacia in bacias:
    slug_name = slugify(bacia)
    slug_names.append(f'{slug_name}-acc.png')
    # print(slug_name)

In [76]:
slug_names = ['bacia-do-rio-branco-acc.png',
 'bacia-do-rio-negro-acc.png',
 'bacia-do-rio-maranon-acc.png',
 'bacia-do-rio-ucayali-acc.png',
 'bacia-do-rio-napo-acc.png',
 'curso-principal-do-rio-amazonas-peru-acc.png',
 'bacia-do-rio-javari-acc.png',
 'bacia-do-rio-ica-putumayo-acc.png',
 'bacia-do-rio-jutai-acc.png',
 'bacia-do-rio-jurua-acc.png',
 'bacia-do-rio-japura-caqueta-acc.png',
 'bacia-do-rio-tefe-acc.png',
 'bacia-do-rio-purus-acc.png',
 'curso-principal-do-rio-solimoes-acc.png',
 'bacia-do-rio-coari-acc.png',
 'bacia-dos-rios-beni-e-madre-de-dios-acc.png',
 'bacia-do-rio-mamore-acc.png',
 'bacia-do-rio-guapore-itenez-acc.png',
 'bacia-do-rio-ji-parana-acc.png',
 'bacia-do-rio-aripuana-acc.png',
 'bacia-do-rio-madeira-acc.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-amazonas-acc.png',
 'bacia-do-rio-abacaxis-acc.png',
 'bacia-do-rio-juruena-acc.png',
 'bacia-do-rio-teles-pires-acc.png',
 'bacia-do-rio-tapajos-acc.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-noroeste-do-para-acc.png',
 'bacia-do-rio-curua-una-acc.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-nordeste-do-pa-acc.png',
 'bacia-do-rio-iriri-acc.png',
 'bacia-do-rio-xingu-acc.png',
 'curso-principal-do-rio-amazonas-brasil-acc.png']

In [75]:
slug_names[14]

'curso-principal-do-rio-solimoes-acc.png'

In [62]:
for i in slug_names:
    print(i)

bacia-do-rio-branco-acc.png
bacia-do-rio-negro-acc.png
bacia-do-rio-maranon-acc.png
bacia-do-rio-ucayali-acc.png
bacia-do-rio-napo-acc.png
curso-principal-do-rio-amazonas-peru-acc.png
bacia-do-rio-javari-acc.png
bacia-do-rio-ica-putumayo-acc.png
bacia-do-rio-jutai-acc.png
bacia-do-rio-jurua-acc.png
bacia-do-rio-japura-caqueta-acc.png
bacia-do-rio-tefe-acc.png
bacia-do-rio-coari-acc.png
bacia-do-rio-purus-acc.png
curso-principal-do-rio-solimoes-acc.png
bacia-dos-rios-beni-e-madre-de-dios-acc.png
bacia-do-rio-mamore-acc.png
bacia-do-rio-guapore-itenez-acc.png
bacia-do-rio-ji-parana-acc.png
bacia-do-rio-aripuana-acc.png
bacia-do-rio-madeira-acc.png
bacias-da-margem-esquerda-do-rio-amazonas-amazonas-acc.png
bacia-do-rio-abacaxis-acc.png
bacia-do-rio-juruena-acc.png
bacia-do-rio-teles-pires-acc.png
bacia-do-rio-tapajos-acc.png
bacias-da-margem-esquerda-do-rio-amazonas-noroeste-do-para-acc.png
bacia-do-rio-curua-una-acc.png
bacias-da-margem-esquerda-do-rio-amazonas-nordeste-do-pa-acc.png
bac

In [83]:
pdfs[2:]

['BHA_20220120.pdf',
 'BHA_20220127.pdf',
 'BHA_20220203.pdf',
 'BHA_20220210.pdf',
 'BHA_20220217.pdf',
 'BHA_20220224.pdf',
 'BHA_20220303.pdf',
 'BHA_20220310.pdf',
 'BHA_20220317.pdf',
 'BHA_20220324.pdf',
 'BHA_20220331.pdf',
 'BHA_20220407.pdf',
 'BHA_20220414.pdf',
 'BHA_20220421.pdf',
 'BHA_20220428.pdf',
 'BHA_20220505.pdf',
 'BHA_20220512.pdf',
 'BHA_20220519.pdf',
 'BHA_20220526.pdf',
 'BHA_20220602.pdf',
 'BHA_20220609.pdf',
 'BHA_20220616.pdf',
 'BHA_20220623.pdf',
 'BHA_20220630.pdf',
 'BHA_20220707.pdf',
 'BHA_20220714.pdf',
 'BHA_20220721.pdf',
 'BHA_20220728.pdf',
 'BHA_20220804.pdf',
 'BHA_20220811.pdf',
 'BHA_20220818.pdf',
 'BHA_20220825.pdf',
 'BHA_20220901.pdf',
 'BHA_20220908.pdf',
 'BHA_20220915.pdf',
 'BHA_20220922.pdf',
 'BHA_20220929.pdf',
 'BHA_20221006.pdf',
 'BHA_20221013.pdf',
 'BHA_20221020.pdf',
 'BHA_20221027.pdf',
 'BHA_20221103.pdf',
 'BHA_20221110.pdf',
 'BHA_20221117.pdf',
 'BHA_20221124.pdf',
 'BHA_20221201.pdf',
 'BHA_20221208.pdf',
 'BHA_2022121

In [85]:
def img_analysis(doc, output_path, slug_names):
    
    path = f"{output_path}/analysis"
    c = 0
    for i in range(4, 15):
        page = doc.load_page(i)
        d = page.get_text("dict")
        blocks = d["blocks"] 
        imgblocks = [b for b in blocks if b["type"] == 1]
        for i in imgblocks:
            if i['height'] < 3000 and i['height'] > 300:
                img = i['image']
                img_name = slug_names[c]
                # img_name = f'{c}.png'
                with open(f'{path}/{img_name}', "wb") as f:
                                f.write(img)
                c += 1
                
path_dir = 'data/pdf/2022'
pdfs = sorted(os.listdir(path_dir))
for pdf in pdfs[2:]:
    mmdd = pdf[8:12]
    full_path = f'{path_dir}/{pdf}'
    doc = pymupdf.open(full_path)
    output_path = f"data/images/2022/{mmdd}"
    img_analysis(doc, output_path, slug_names)
    print(mmdd)

0120
0127
0203
0210
0217
0224
0303
0310
0317
0324
0331
0407
0414
0421
0428
0505
0512
0519
0526
0602
0609
0616
0623
0630
0707
0714
0721
0728
0804
0811
0818
0825
0901
0908
0915
0922
0929
1006
1013
1020
1027
1103
1110
1117
1124
1201
1208
1215
1222
1229


In [86]:
path_dir = 'data/pdf/2022'
pdfs = sorted(os.listdir(path_dir))

In [99]:
pdfs[49:]

['BHA_20221215.pdf', 'BHA_20221222.pdf', 'BHA_20221229.pdf']

In [100]:
def img_anomaly(doc, output_path):
    c = 1
    path = f"{output_path}/anomaly"
    for p in range(19,24):
        page = doc.load_page(p)
        d = page.get_text("dict")
        blocks = d["blocks"] 
        imgblocks = [b for b in blocks if b["type"] == 1]
        for i in imgblocks:
            if i['height'] > 250:
                bbox = i['bbox']
                rect = pymupdf.Rect(bbox)
                zoom = 3
                mat = pymupdf.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
                pix.save(f'{path}/bacia_{c}.png')
                c += 1
                
for pdf in pdfs[49:]:
    mmdd = pdf[8:12]
    full_path = f'{path_dir}/{pdf}'
    doc = pymupdf.open(full_path)
    output_path = f"data/images/2022/{mmdd}"
    img_anomaly(doc, output_path)
    print(mmdd)

1215
1222
1229


In [106]:
pdfs[2:]

['BHA_20220120.pdf',
 'BHA_20220127.pdf',
 'BHA_20220203.pdf',
 'BHA_20220210.pdf',
 'BHA_20220217.pdf',
 'BHA_20220224.pdf',
 'BHA_20220303.pdf',
 'BHA_20220310.pdf',
 'BHA_20220317.pdf',
 'BHA_20220324.pdf',
 'BHA_20220331.pdf',
 'BHA_20220407.pdf',
 'BHA_20220414.pdf',
 'BHA_20220421.pdf',
 'BHA_20220428.pdf',
 'BHA_20220505.pdf',
 'BHA_20220512.pdf',
 'BHA_20220519.pdf',
 'BHA_20220526.pdf',
 'BHA_20220602.pdf',
 'BHA_20220609.pdf',
 'BHA_20220616.pdf',
 'BHA_20220623.pdf',
 'BHA_20220630.pdf',
 'BHA_20220707.pdf',
 'BHA_20220714.pdf',
 'BHA_20220721.pdf',
 'BHA_20220728.pdf',
 'BHA_20220804.pdf',
 'BHA_20220811.pdf',
 'BHA_20220818.pdf',
 'BHA_20220825.pdf',
 'BHA_20220901.pdf',
 'BHA_20220908.pdf',
 'BHA_20220915.pdf',
 'BHA_20220922.pdf',
 'BHA_20220929.pdf',
 'BHA_20221006.pdf',
 'BHA_20221013.pdf',
 'BHA_20221020.pdf',
 'BHA_20221027.pdf',
 'BHA_20221103.pdf',
 'BHA_20221110.pdf',
 'BHA_20221117.pdf',
 'BHA_20221124.pdf',
 'BHA_20221201.pdf',
 'BHA_20221208.pdf',
 'BHA_2022121

In [118]:
doc = pymupdf.open('data/pdf/2022/BHA_20220106.pdf')
output_path = 'test'

def img_reference(doc, output_path):
    path = f"{output_path}/reference"
    page = doc.load_page(17)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if 'Abacaxis' in text:
            break
    rect = pymupdf.Rect(x0, y0, x1, y1) # type: ignore
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/reference_table.png')
    

for pdf in pdfs[49:]:
    mmdd = pdf[8:12]
    full_path = f'{path_dir}/{pdf}'
    doc = pymupdf.open(full_path)
    output_path = f"data/images/2022/{mmdd}"
    img_reference(doc, output_path)
    print(mmdd)

1215
1222
1229


In [ ]:
def img_categorization(doc, output_path):
    path = f"{output_path}/categorization"
    page = doc.load_page(18)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if 'Abacaxis' in text:
            break
    rect = pymupdf.Rect(x0, y0, x1, y1) # type: ignore
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/table.png')
    

for pdf in pdfs[49:]:
    mmdd = pdf[8:12]
    full_path = f'{path_dir}/{pdf}'
    doc = pymupdf.open(full_path)
    output_path = f"data/images/2022/{mmdd}"
    img_categorization(doc, output_path)
    print(mmdd)

1215
1222
1229


In [122]:
pdfs[9:]

['BHA_20220310.pdf',
 'BHA_20220317.pdf',
 'BHA_20220324.pdf',
 'BHA_20220331.pdf',
 'BHA_20220407.pdf',
 'BHA_20220414.pdf',
 'BHA_20220421.pdf',
 'BHA_20220428.pdf',
 'BHA_20220505.pdf',
 'BHA_20220512.pdf',
 'BHA_20220519.pdf',
 'BHA_20220526.pdf',
 'BHA_20220602.pdf',
 'BHA_20220609.pdf',
 'BHA_20220616.pdf',
 'BHA_20220623.pdf',
 'BHA_20220630.pdf',
 'BHA_20220707.pdf',
 'BHA_20220714.pdf',
 'BHA_20220721.pdf',
 'BHA_20220728.pdf',
 'BHA_20220804.pdf',
 'BHA_20220811.pdf',
 'BHA_20220818.pdf',
 'BHA_20220825.pdf',
 'BHA_20220901.pdf',
 'BHA_20220908.pdf',
 'BHA_20220915.pdf',
 'BHA_20220922.pdf',
 'BHA_20220929.pdf',
 'BHA_20221006.pdf',
 'BHA_20221013.pdf',
 'BHA_20221020.pdf',
 'BHA_20221027.pdf',
 'BHA_20221103.pdf',
 'BHA_20221110.pdf',
 'BHA_20221117.pdf',
 'BHA_20221124.pdf',
 'BHA_20221201.pdf',
 'BHA_20221208.pdf',
 'BHA_20221215.pdf',
 'BHA_20221222.pdf',
 'BHA_20221229.pdf']

In [126]:
def img_multimodel(doc, output_path):
    page = doc.load_page(15)
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    c = 0
    img_name = ["seven_days", "fourteen_days"]
    for i in imgblocks:
        if i['height'] == 800:
            img = i['image']
            name = img_name[c]
            with open(f'{output_path}/multimodel/{name}.png', "wb") as f:
                f.write(img)
    
            c += 1
            
# doc = pymupdf.open('data/pdf/2022/BHA_20220310.pdf')
# output_path = 'test'

for pdf in pdfs[49:]:
    mmdd = pdf[8:12]
    full_path = f'{path_dir}/{pdf}'
    doc = pymupdf.open(full_path)
    output_path = f"data/images/2022/{mmdd}"
    img_multimodel(doc, output_path)
    print(mmdd)

1215
1222
1229


In [127]:
from cloudinary import uploader, config, api
import json

In [128]:
config(
  cloud_name="dveg6vhbi",
  api_key="954532419258163",
  api_secret="MiQDehQG2afKCxVi-QC8qMFE0Ag"
)

In [129]:
yyyy = 2022 
for pdf in pdfs:
    mmdd = pdf[8:12]
    
    pathPT = f'{path_dir}/{pdf}'
    d_imgs = {}
    output_path = f'data/images/{yyyy}/{mmdd}'
    for section in os.listdir(output_path):
        full_path = f'{output_path}/{section}'
        d_imgs[section] = {}
        imgs = os.listdir(full_path)
        for img in imgs:
            full_img_path = f'{full_path}/{img}'
            img_name = img.split(".")[0]
            result = uploader.upload(
                full_img_path,
                public_id=img_name,
                folder=f"clima-amazonia/{yyyy}/{mmdd}/{section}",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
            url = result['secure_url']
            d_imgs[section][img_name] = url
            print(result)
    print(output_path)
    
    pdfs = {}
    result = uploader.upload(
                pathPT,
                public_id=pathPT.split("/")[-1].replace(".pdf",""),
                folder=f"clima-amazonia/{yyyy}/{mmdd}/",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
    pdfs['pt'] = result['secure_url']
    
    jsonPath = f'./data/{yyyy}/{mmdd}.json'
    with open(jsonPath, 'r') as f:
        bulletin_dict = json.load(f)
    bulletin_dict['images'] = d_imgs
    bulletin_dict['pdf'] = pdfs
    with open(jsonPath, 'w') as f:
            json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)

{'asset_id': '6720d9746877696cfd4d8818ae2e3646', 'public_id': 'clima-amazonia/2022/0106/current_conditions/table_current_conditions', 'version': 1774646049, 'version_id': '754097282e6ada192659b7e8e2b1b27f', 'signature': '6b1e4b1f5896db8c9aa6676a72a9981e76f97307', 'width': 1429, 'height': 274, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-27T21:14:09Z', 'tags': [], 'bytes': 71203, 'type': 'upload', 'etag': '5c9899e7a369200d38d6ecf6234b4488', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1774646049/clima-amazonia/2022/0106/current_conditions/table_current_conditions.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774646049/clima-amazonia/2022/0106/current_conditions/table_current_conditions.png', 'asset_folder': 'clima-amazonia/2022/0106/current_conditions', 'display_name': 'table_current_conditions', 'original_filename': 'table_current_conditions', 'api_key': '954532419258163'}
{'asset_id': 'f2318140e35e51d93

In [ ]:
path = "data/pdf/2022"
for pdf in sorted(os.listdir(path)):
    mmdd = pdf[8:12]
    path_pdf = f"{path}/{pdf}"
    doc = pymupdf.open(path_pdf)
    page = doc.load_page(0)
    pix = page.get_pixmap()
    pix.save(f"test/{mmdd}.png")
    
    print(mmdd)

In [3]:
path = "data/2022"
eds = sorted(os.listdir(path))
d_n = {}
c = 1
for i in eds:
    mmdd = i.split(".")[0]
    mm = mmdd[:2]
    if mm in d_n:
        d_i = {
            "number": c,
            "url": f"2023/{mmdd}",
            "img": ""
        }
        d_n[mm]["issues"].append(d_i)
    else:
        d_n[mm] = {
             "month": mm,
             "issues": []
        }
        d_i = {
            "number": c,
            "url": f"2024/{mmdd}",
            "img": ""
        }
        d_n[mm]["issues"].append(d_i)
    c += 1
    print(mmdd)

0106
0113
0120
0127
0203
0210
0217
0224
0303
0310
0317
0324
0331
0407
0414
0421
0428
0505
0512
0519
0526
0602
0609
0616
0623
0630
0707
0714
0721
0728
0804
0811
0818
0825
0901
0908
0915
0922
0929
1006
1013
1020
1027
1103
1110
1117
1124
1201
1208
1215
1222
1229


In [4]:
ns = [v for v in d_n.values()]

In [6]:
ns.reverse()

In [11]:
ns

[{'month': '12',
  'issues': [{'number': 48,
    'url': '2024/1201',
    'img': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890686/clima-amazonia/2023/1201/1201.png'},
   {'number': 49,
    'url': '2023/1208',
    'img': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890687/clima-amazonia/2023/1208/1208.png'},
   {'number': 50,
    'url': '2023/1215',
    'img': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890688/clima-amazonia/2023/1215/1215.png'},
   {'number': 51,
    'url': '2023/1222',
    'img': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890689/clima-amazonia/2023/1222/1222.png'},
   {'number': 52,
    'url': '2023/1229',
    'img': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890690/clima-amazonia/2023/1229/1229.png'}]},
 {'month': '11',
  'issues': [{'number': 44,
    'url': '2024/1103',
    'img': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890692/clima-amazonia/2023/1103/1103.png'},
   {'number': 45,
    'url

In [12]:
with open("data/2022/issues.json", "w") as file:
    json.dump(ns, file, ensure_ascii=False, indent=3)

In [8]:
from cloudinary import uploader, config, api

In [9]:
config(
  cloud_name="dveg6vhbi",
  api_key="954532419258163",
  api_secret="MiQDehQG2afKCxVi-QC8qMFE0Ag"
)

In [10]:
for n in ns:
    issues = n['issues']
    for i in issues:
        url = i['url']
        mmdd = url.split("/")[1]
        path_img = f"test/{mmdd}.png"
        result = uploader.upload(
                path_img,
                public_id=mmdd,
                folder=f"clima-amazonia/2023/{mmdd}",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
        secure_url = result['secure_url']
        i['img'] = secure_url
        print(result)

{'asset_id': '9222c058e1db9087b839a19c3d354cfe', 'public_id': 'clima-amazonia/2023/1201/1201', 'version': 1774890686, 'version_id': '3e578e8c2f51b66d1de42bdc9548655e', 'signature': '19be729e78960d4aba8dd22a15654b0c3ce6e308', 'width': 612, 'height': 792, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-30T17:11:26Z', 'tags': [], 'bytes': 61294, 'type': 'upload', 'etag': '9b439958ed2db386c635769dc5a68bc5', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1774890686/clima-amazonia/2023/1201/1201.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774890686/clima-amazonia/2023/1201/1201.png', 'asset_folder': 'clima-amazonia/2023/1201', 'display_name': '1201', 'original_filename': '1201', 'api_key': '954532419258163'}
{'asset_id': '2f7190b4fbca15bb93c843db470bbfa6', 'public_id': 'clima-amazonia/2023/1208/1208', 'version': 1774890687, 'version_id': 'e172548be48a0e5119e1627712a23f7c', 'signature': '0197fc9582ef7f09a79ca663de

In [13]:
with open("/home/inacio/clima-amazonia/data/issues.json", "r") as file:
    json_data = json.load(file)

In [24]:
for i in json_data[-1]['numbers']:
    issues = i['issues']
    for issue in issues:
        url = issue['url']
        year = url.split("/")[0]
        mmdd = url.split("/")[1]
        issue['url'] = f'2022/{mmdd}'

In [25]:
with open("/home/inacio/clima-amazonia/data/issues.json", "w") as file:
    json.dump(json_data, file, ensure_ascii=False, indent=3)